In [1]:
#-----------------------------------------------
# FLIGHT-DELAY-ANALYSIS
#-----------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')


# Load Dataset (Block 1)
df = pd.read_csv('../data/raw/flights_sample_3m.csv', low_memory=False)

# Quick Audit 
print('Shape:', df.shape)
print('\nCloumns:\n', df.columns.tolist())
print('\nData Types:\n', df.dtypes)
print('\nFirst 5 rows\n:', df.head())

Shape: (3000000, 32)

Cloumns:
 ['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']

Data Types:
 FL_DATE                     object
AIRLINE                     object
AIRLINE_DOT                 object
AIRLINE_CODE                object
DOT_CODE                     int64
FL_NUMBER                    int64
ORIGIN                      object
ORIGIN_CITY                 object
DEST                        object
DEST_CITY                   object
CRS_DEP_TIME                 int64
DEP_TIME                   float64
DEP_DELAY                  float64
TAXI_OUT                   f

In [2]:
# Missing value analysis(Block 2)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print("Missing Value Report:")
print(missing_report)

Missing Value Report:
                         Missing Count  Missing %
CANCELLATION_CODE              2920860      97.36
DELAY_DUE_CARRIER              2466137      82.20
DELAY_DUE_SECURITY             2466137      82.20
DELAY_DUE_NAS                  2466137      82.20
DELAY_DUE_WEATHER              2466137      82.20
DELAY_DUE_LATE_AIRCRAFT        2466137      82.20
ARR_DELAY                        86198       2.87
ELAPSED_TIME                     86198       2.87
AIR_TIME                         86198       2.87
ARR_TIME                         79942       2.66
TAXI_IN                          79944       2.66
WHEELS_ON                        79944       2.66
WHEELS_OFF                       78806       2.63
TAXI_OUT                         78806       2.63
DEP_DELAY                        77644       2.59
DEP_TIME                         77615       2.59
CRS_ELAPSED_TIME                    14       0.00


In [3]:
# Smart cleaning(Block 3)
# Step 1: Fix date column
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
print(df['FL_DATE'].dtype)

datetime64[ns]


In [4]:
# Step 2: Fill delay cause NaNs with 0
# (NaN = no delay from that cause, NOT missing data)
delay_cause_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
                    'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 
                    'DELAY_DUE_LATE_AIRCRAFT']
df[delay_cause_cols] = df[delay_cause_cols].fillna(0)

print("Delay columns filled. Sample:")
print(df[delay_cause_cols].head())

Delay columns filled. Sample:
   DELAY_DUE_CARRIER  DELAY_DUE_WEATHER  DELAY_DUE_NAS  DELAY_DUE_SECURITY  \
0                0.0                0.0            0.0                 0.0   
1                0.0                0.0            0.0                 0.0   
2                0.0                0.0            0.0                 0.0   
3                0.0                0.0           24.0                 0.0   
4                0.0                0.0            0.0                 0.0   

   DELAY_DUE_LATE_AIRCRAFT  
0                      0.0  
1                      0.0  
2                      0.0  
3                      0.0  
4                      0.0  


In [5]:
# Step 3: Drop rows where ARR_DELAY is null AND flight wasn't cancelled, genuine data errors
before = len(df)
df = df[~(df['ARR_DELAY'].isnull() & (df['CANCELLED'] == 0))]
after = len(df)
print(f"Rows dropped (data errors): {before - after:,}")

Rows dropped (data errors): 7,058


In [6]:
# Step 4: Drop the 14 rows with missing CRS_ELAPSED_TIME
df = df.dropna(subset=['CRS_ELAPSED_TIME'])
print(f"After dropping 14 bad rows:{len(df):,}")

After dropping 14 bad rows:2,992,928


In [7]:
#step 5: Remove Duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df):,}")

print(f"\nFinal clean shape: {df.shape}")
print("\nMissing values in key columns:")
print(df[['ARR_DELAY','DEP_DELAY','DELAY_DUE_CARRIER','CANCELLED']].isnull().sum())

Duplicates removed: 0

Final clean shape: (2992928, 32)

Missing values in key columns:
ARR_DELAY            79126
DEP_DELAY            77630
DELAY_DUE_CARRIER        0
CANCELLED                0
dtype: int64


In [11]:
# -------- FEATURE ENGINEERING -------------------------

# 1. Extract date components for grouping later
df['Month']      = df['FL_DATE'].dt.month
df['Month_Name'] = df['FL_DATE'].dt.strftime('%b')
df['Day_Name']   = df['FL_DATE'].dt.strftime('%A')
df['Quarter']    = df['FL_DATE'].dt.quarter
df['Year']       = df['FL_DATE'].dt.year

# 2. Is the flight delayed? (industry standard = 15+ mins)
df['Is_Delayed'] = (df['ARR_DELAY'] >= 15).astype(int)

# 3. Delay severity bucket
def classify_delay(mins):
    if pd.isna(mins) or mins < 15:
        return 'On Time'
    elif mins < 30:
        return 'minor (15-30 min)'
    elif mins < 60:
        return 'Moderate (30-60 min)'
    elif mins < 120:
        return 'Severe (1-2 hrs)'
    else:
        return 'Critical (2+ hrs)'

df['Delay_Category'] = df['ARR_DELAY'].apply(classify_delay)

# 4. Primary delay cause per flight
def primary_cause(row):
    causes = {
        'Carrier': row['DELAY_DUE_CARRIER'],
        'Weather': row['DELAY_DUE_WEATHER'],
        'NAS/ATC': row['DELAY_DUE_NAS'],
        'Security': row['DELAY_DUE_SECURITY'],
        'Late Aircraft': row['DELAY_DUE_LATE_AIRCRAFT']
    }

    if sum(causes.values()) == 0:
        return 'No Delay'
    return max(causes, key=causes.get)
    
df['Primary_Cause'] = df.apply(primary_cause, axis=1)

# 5. Estimated cost (carrier delays only — $74/min industry rate)
COST_PER_MIN = 74
df['Estimated_Cost_USD'] = df['DELAY_DUE_CARRIER'] * COST_PER_MIN

print("Feature engineering complete!")
print(f"\nNew columns added: Month, Month_Name, Day_Name, Quarter, Year,")
print(f"Is_Delayed, Delay_Category, Primary_Cause, Estimated_Cost_USD")
print(f"\nDelay category breakdown:")
print(df['Delay_Category'].value_counts())
print(f"\nPrimary cause breakdown:")
print(df['Primary_Cause'].value_counts())

Feature engineering complete!

New columns added: Month, Month_Name, Day_Name, Quarter, Year,
Is_Delayed, Delay_Category, Primary_Cause, Estimated_Cost_USD

Delay category breakdown:
Delay_Category
On Time                 2459065
minor (15-30 min)        194171
Moderate (30-60 min)     161280
Severe (1-2 hrs)         108611
Critical (2+ hrs)         69801
Name: count, dtype: int64

Primary cause breakdown:
Primary_Cause
No Delay         2459065
Late Aircraft     194789
Carrier           179927
NAS/ATC           137192
Weather            20350
Security            1605
Name: count, dtype: int64


In [12]:
# Save clean dataset
df.to_csv('../data/clean/flights_clean.csv', index=False)
print(f"Saved! Rows: {len(df):,} | Columns: {len(df.columns)}")
print("\nColumns in clean file:")
print(df.columns.tolist())

Saved! Rows: 2,992,928 | Columns: 41

Columns in clean file:
['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'Month', 'Month_Name', 'Day_Name', 'Quarter', 'Year', 'Is_Delayed', 'Delay_Category', 'Estimated_Cost_USD', 'Primary_Cause']
